# LAB | Extractive Question Answering

This notebook demonstrates how Pinecone helps you build an extractive question-answering application using:
- A **vector index** (Pinecone) to store and run semantic search
- A **retriever** model for embedding context passages
- A **reader** model to extract answers

We use the SQuAD dataset which contains questions and context paragraphs with answers.

## ⚙️ Step 1 — Install Dependencies

> **Important:** We do NOT reinstall `torch` — we use Colab's pre-installed version to avoid torchvision conflicts.

In [1]:
# Install only what we need — do NOT reinstall torch
!pip install -q sentence-transformers datasets pinecone transformers tqdm python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.9 MB/s eta 0:00:00


## 🔑 Step 2 — Set API Keys

Get a free Pinecone API key at https://app.pinecone.io and paste it below.

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

print("Key loaded:", OPENAI_API_KEY is not None)


Enter your OpenAI API key: ··········
Key loaded: True


In [5]:
import os
from google.colab import userdata
from pinecone import Pinecone, ServerlessSpec

# Get API key from Colab secrets
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')

# connect to pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "extractive-question-answering"
spec = ServerlessSpec(cloud="aws", region="eu-west-1")  # Frankfurt, Europe

# create the index if it does not exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

# connect to the index
index = pc.Index(index_name)
index.describe_index_stats()

Index 'extractive-question-answering' already exists.


DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)

## 📦 Step 3 — Load Dataset

Load the SQuAD dataset from HuggingFace, keep only the `title` and `context` columns, and drop duplicate contexts.

In [3]:
from datasets import load_dataset
import pandas as pd

# load the squad dataset into a pandas dataframe
df = load_dataset("squad", split="train").to_pandas()

# select only title and context column
df = df[['title', 'context']]
# drop rows containing duplicate context passages
df = df.drop_duplicates(subset='context').reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
df.head()


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

Dataset shape: (18891, 2)


,title,context
0,University_of_Notre_Dame,"Architecturally, the school has a Catholic cha..."
1,University_of_Notre_Dame,"As at most other universities, Notre Dame's st..."
2,University_of_Notre_Dame,The university is the major seat of the Congre...
3,University_of_Notre_Dame,The College of Engineering was established in ...
4,University_of_Notre_Dame,All of Notre Dame's undergraduate students are...


## 🌲 Step 4 — Initialize Pinecone Index

Create a Pinecone index named `extractive-question-answering` with:
- **dimension = 384** (matches our retriever model output)
- **metric = cosine** (the retriever is optimized for cosine similarity)

In [6]:
from pinecone import Pinecone, ServerlessSpec

# connect to pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "extractive-question-answering"
spec = ServerlessSpec(cloud="aws", region="us-east-1")

# create the index if it does not exist
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=spec
    )
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")

# connect to the index
index = pc.Index(index_name)
index.describe_index_stats()


Index 'extractive-question-answering' already exists.


DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)

## 🔍 Step 5 — Initialize Retriever

Load the `multi-qa-MiniLM-L6-cos-v1` SentenceTransformer model. It outputs 384-dimensional vectors and is specifically trained on 215M (question, answer) pairs for semantic search.

In [7]:
import torch
from sentence_transformers import SentenceTransformer

# set device to GPU if available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# load the retriever model from huggingface model hub
retriever = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')
# move to device manually to avoid torchvision conflicts
retriever = retriever.to(device)
print(retriever)


Using device: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


## 📤 Step 6 — Generate Embeddings and Upsert to Pinecone

Encode all context passages in batches of 64 and upload them to Pinecone. Each record contains:
- A unique string **ID**
- The **embedding** vector (384 dims)
- **Metadata**: title and context text (so we can retrieve the text later)

In [8]:
from tqdm.auto import tqdm

# we will use batches of 64
batch_size = 64

for i in tqdm(range(0, len(df), batch_size)):
    # find end of batch
    i_end = min(i + batch_size, len(df))
    # extract batch
    batch = df.iloc[i:i_end]
    # generate embeddings for batch
    emb = retriever.encode(batch['context'].tolist()).tolist()
    # get metadata
    meta = batch[['title', 'context']].to_dict(orient='records')
    # create unique IDs
    ids = [str(x) for x in range(i, i_end)]
    # add all to upsert list
    to_upsert = list(zip(ids, emb, meta))
    # upsert/insert these records to pinecone
    _ = index.upsert(vectors=to_upsert)

# check that we have all vectors in index
index.describe_index_stats()


  0%|          | 0/296 [00:00<?, ?it/s]

DescribeIndexStatsResponse(dimension=384, total_vector_count=18891, metric='cosine', namespaces=1)

## 📖 Step 7 — Initialize Reader

Load `deepset/electra-base-squad2` into a HuggingFace `question-answering` pipeline. This model is fine-tuned on SQuAD2 and extracts the exact answer span from a given context.

In [9]:
from transformers import pipeline

model_name = 'deepset/electra-base-squad2'

# Use device index: 0 for GPU, -1 for CPU
device_id = 0 if device == 'cuda' else -1

# load the reader model into a question-answering pipeline
reader = pipeline(
    tokenizer=model_name,
    model=model_name,
    task='question-answering',
    device=device_id
)
print(reader)


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

QuestionAnsweringPipeline: {'model': 'ElectraForQuestionAnswering', 'dtype': 'float32', 'device': 'cuda', 'input_modalities': 'text'}


## 🛠️ Step 8 — Helper Functions

- `get_context(question, top_k)`: encodes the question and retrieves the top-k most similar context passages from Pinecone
- `extract_answer(question, context)`: runs each context through the reader and returns results sorted by confidence score

In [10]:
# gets context passages from the pinecone index
def get_context(question, top_k):
    # generate embeddings for the question
    xq = retriever.encode([question]).tolist()
    # search pinecone index for context passage with the answer
    xc = index.query(vector=xq, top_k=top_k, include_metadata=True)
    # extract the context passage from pinecone search result
    c = [match['metadata']['context'] for match in xc['matches']]
    return c


In [11]:
from pprint import pprint

# extracts answer from the context passage
def extract_answer(question, context):
    results = []
    for c in context:
        # feed the reader the question and context to extract an answer
        answer = reader(question=question, context=c)
        # add the context to answer dict for printing both together
        answer['context'] = c
        results.append(answer)
    # sort the results based on the score from reader model (highest first)
    sorted_result = sorted(results, key=lambda x: x['score'], reverse=True)
    pprint(sorted_result)
    return sorted_result


## ❓ Step 9 — Run Queries

Let's test the full pipeline with several questions.

In [12]:
# Query 1
question = "How much oil is Egypt producing in a day?"
context = get_context(question, top_k=1)
print("Retrieved context:")
print(context)


Retrieved context:
['Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.']


In [13]:
extract_answer(question, context)


[{'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of '
             'natural gas (in 2013), which makes Egypt as the largest oil '
             'producer not member of the Organization of the Petroleum '
             'Exporting Countries (OPEC) and the second-largest dry natural '
             'gas producer in Africa. In 2013, Egypt was the largest consumer '
             'of oil and natural gas in Africa, as more than 20% of total oil '
             'consumption and more than 40% of total dry natural gas '
             'consumption in Africa. Also, Egypt possesses the largest oil '
             'refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is '
             'currently planning to build its first nuclear power plant in El '
             'Dabaa city, northern Egypt.',
  'end': 33,
  'score': 0.9999855201981802,
  'start': 20}]


[{'score': 0.9999855201981802,
  'start': 20,
  'end': 33,
  'answer': '691,000 bbl/d',
  'context': 'Egypt was producing 691,000 bbl/d of oil and 2,141.05 Tcf of natural gas (in 2013), which makes Egypt as the largest oil producer not member of the Organization of the Petroleum Exporting Countries (OPEC) and the second-largest dry natural gas producer in Africa. In 2013, Egypt was the largest consumer of oil and natural gas in Africa, as more than 20% of total oil consumption and more than 40% of total dry natural gas consumption in Africa. Also, Egypt possesses the largest oil refinery capacity in Africa 726,000 bbl/d (in 2012). Egypt is currently planning to build its first nuclear power plant in El Dabaa city, northern Egypt.'}]

In [14]:
# Query 2
question = "What are the first names of the men that invented YouTube?"
context = get_context(question, top_k=1)
extract_answer(question, context)


[{'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, '
             'Hurley and Chen developed the idea for YouTube during the early '
             'months of 2005, after they had experienced difficulty sharing '
             "videos that had been shot at a dinner party at Chen's apartment "
             'in San Francisco. Karim did not attend the party and denied that '
             'it had occurred, but Chen commented that the idea that YouTube '
             'was founded after a dinner party "was probably very strengthened '
             'by marketing ideas around creating a story that was very '
             'digestible".',
  'end': 79,
  'score': 0.9999276399612427,
  'start': 64}]


[{'score': 0.9999276399612427,
  'start': 64,
  'end': 79,
  'answer': 'Hurley and Chen',
  'context': 'According to a story that has often been repeated in the media, Hurley and Chen developed the idea for YouTube during the early months of 2005, after they had experienced difficulty sharing videos that had been shot at a dinner party at Chen\'s apartment in San Francisco. Karim did not attend the party and denied that it had occurred, but Chen commented that the idea that YouTube was founded after a dinner party "was probably very strengthened by marketing ideas around creating a story that was very digestible".'}]

In [15]:
# Query 3
question = "What is Albert Einstein famous for?"
context = get_context(question, top_k=1)
extract_answer(question, context)


[{'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity '
             'and general relativity. He also made important contributions to '
             'statistical mechanics, especially his mathematical treatment of '
             'Brownian motion, his resolution of the paradox of specific '
             'heats, and his connection of fluctuations and dissipation. '
             'Despite his reservations about its interpretation, Einstein also '
             'made contributions to quantum mechanics and, indirectly, quantum '
             'field theory, primarily through his theoretical studies of the '
             'photon.',
  'end': 86,
  'score': 0.9851929545402527,
  'start': 29}]


[{'score': 0.9851929545402527,
  'start': 29,
  'end': 86,
  'answer': 'his theories of special relativity and general relativity',
  'context': 'Albert Einstein is known for his theories of special relativity and general relativity. He also made important contributions to statistical mechanics, especially his mathematical treatment of Brownian motion, his resolution of the paradox of specific heats, and his connection of fluctuations and dissipation. Despite his reservations about its interpretation, Einstein also made contributions to quantum mechanics and, indirectly, quantum field theory, primarily through his theoretical studies of the photon.'}]

In [16]:
# Query 4 — top 3 contexts
question = "Who was the first person to step foot on the moon?"
context = get_context(question, top_k=3)
extract_answer(question, context)


[{'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving '
             'orbit, Armstrong and Aldrin transferred into the Lunar Module, '
             'named Eagle, and after a landing gear inspection by Collins '
             'remaining in the Command/Service Module Columbia, began their '
             'descent. After overcoming several computer overload alarms '
             'caused by an antenna switch left in the wrong position, and a '
             'slight downrange error, Armstrong took over manual flight '
             'control at about 180 meters (590 ft), and guided the Lunar '
             'Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 '
             '(3:17:04 pm CDT). The first humans on the Moon would wait '
             'another six hours before they ventured out of their craft. At '
             '02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the '
             'first human to set foot on the Moon.',

[{'score': 0.9998037815093994,
  'start': 71,
  'end': 80,
  'answer': 'Armstrong',
  'context': 'The trip to the Moon took just over three days. After achieving orbit, Armstrong and Aldrin transferred into the Lunar Module, named Eagle, and after a landing gear inspection by Collins remaining in the Command/Service Module Columbia, began their descent. After overcoming several computer overload alarms caused by an antenna switch left in the wrong position, and a slight downrange error, Armstrong took over manual flight control at about 180 meters (590 ft), and guided the Lunar Module to a safe landing spot at 20:18:04 UTC, July 20, 1969 (3:17:04 pm CDT). The first humans on the Moon would wait another six hours before they ventured out of their craft. At 02:56 UTC, July 21 (9:56 pm CDT July 20), Armstrong became the first human to set foot on the Moon.'},
 {'score': 0.7011036559706554,
  'start': 240,
  'end': 246,
  'answer': 'Aldrin',
  'context': 'The first step was witnessed by at

## ➕ Additional Questions — What Did You Observe?

Here are a few more questions to explore the system. Notice how the retriever finds semantically relevant passages even without exact keyword matches, and the reader pinpoints the exact answer span.

In [17]:
# Extra question 1
question = "Who wrote the theory of relativity?"
context = get_context(question, top_k=2)
extract_answer(question, context)


[{'answer': 'Einstein',
  'context': 'The beginning of the 20th century brought the start of a '
             'revolution in physics. The long-held theories of Newton were '
             'shown not to be correct in all circumstances. Beginning in 1900, '
             'Max Planck, Albert Einstein, Niels Bohr and others developed '
             'quantum theories to explain various anomalous experimental '
             'results, by introducing discrete energy levels. Not only did '
             'quantum mechanics show that the laws of motion did not hold on '
             'small scales, but even more disturbingly, the theory of general '
             'relativity, proposed by Einstein in 1915, showed that the fixed '
             'background of spacetime, on which both Newtonian mechanics and '
             'special relativity depended, could not exist. In 1925, Werner '
             'Heisenberg and Erwin Schrödinger formulated quantum mechanics, '
             'which explained the precedi

[{'score': 0.9999198913574219,
  'start': 515,
  'end': 523,
  'answer': 'Einstein',
  'context': 'The beginning of the 20th century brought the start of a revolution in physics. The long-held theories of Newton were shown not to be correct in all circumstances. Beginning in 1900, Max Planck, Albert Einstein, Niels Bohr and others developed quantum theories to explain various anomalous experimental results, by introducing discrete energy levels. Not only did quantum mechanics show that the laws of motion did not hold on small scales, but even more disturbingly, the theory of general relativity, proposed by Einstein in 1915, showed that the fixed background of spacetime, on which both Newtonian mechanics and special relativity depended, could not exist. In 1925, Werner Heisenberg and Erwin Schrödinger formulated quantum mechanics, which explained the preceding quantum theories. The observation by Edwin Hubble in 1929 that the speed at which galaxies recede positively correlates with the

In [18]:
# Extra question 2
question = "What country is the Eiffel Tower located in?"
context = get_context(question, top_k=1)
extract_answer(question, context)


[{'answer': 'Eiffel Tower',
  'context': 'The centre of Paris contains the most visited monuments in the '
             'city, including the Notre Dame Cathedral and the Louvre as well '
             'as the Sainte-Chapelle; Les Invalides, where the tomb of '
             'Napoleon is located, and the Eiffel Tower are located on the '
             'Left Bank south-west of the centre. The banks of the Seine from '
             "the Pont de Sully to the Pont d'Iéna have been listed as a "
             'UNESCO World Heritage Site since 1991. Other landmarks are laid '
             'out east to west along the historic axis of Paris, which runs '
             'from the Louvre through the Tuileries Garden, the Luxor Column '
             'in the Place de la Concorde, the Arc de Triomphe, to the Grande '
             'Arche of La Défense.',
  'end': 225,
  'score': 3.310009014967363e-05,
  'start': 213}]


[{'score': 3.310009014967363e-05,
  'start': 213,
  'end': 225,
  'answer': 'Eiffel Tower',
  'context': "The centre of Paris contains the most visited monuments in the city, including the Notre Dame Cathedral and the Louvre as well as the Sainte-Chapelle; Les Invalides, where the tomb of Napoleon is located, and the Eiffel Tower are located on the Left Bank south-west of the centre. The banks of the Seine from the Pont de Sully to the Pont d'Iéna have been listed as a UNESCO World Heritage Site since 1991. Other landmarks are laid out east to west along the historic axis of Paris, which runs from the Louvre through the Tuileries Garden, the Luxor Column in the Place de la Concorde, the Arc de Triomphe, to the Grande Arche of La Défense."}]

In [19]:
# Extra question 3
question = "How many bones does the human body have?"
context = get_context(question, top_k=2)
extract_answer(question, context)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[{'answer': 'very lightweight',
  'context': 'The skeleton consists of very lightweight bones. They have large '
             'air-filled cavities (called pneumatic cavities) which connect '
             'with the respiratory system. The skull bones in adults are fused '
             'and do not show cranial sutures. The orbits are large and '
             'separated by a bony septum. The spine has cervical, thoracic, '
             'lumbar and caudal regions with the number of cervical (neck) '
             'vertebrae highly variable and especially flexible, but movement '
             'is reduced in the anterior thoracic vertebrae and absent in the '
             'later vertebrae. The last few are fused with the pelvis to form '
             'the synsacrum. The ribs are flattened and the sternum is keeled '
             'for the attachment of flight muscles except in the flightless '
             'bird orders. The forelimbs are modified into wings.',
  'end': 41,
  'score': 0.6206583

[{'score': 0.6206583976745605,
  'start': 25,
  'end': 41,
  'answer': 'very lightweight',
  'context': 'The skeleton consists of very lightweight bones. They have large air-filled cavities (called pneumatic cavities) which connect with the respiratory system. The skull bones in adults are fused and do not show cranial sutures. The orbits are large and separated by a bony septum. The spine has cervical, thoracic, lumbar and caudal regions with the number of cervical (neck) vertebrae highly variable and especially flexible, but movement is reduced in the anterior thoracic vertebrae and absent in the later vertebrae. The last few are fused with the pelvis to form the synsacrum. The ribs are flattened and the sternum is keeled for the attachment of flight muscles except in the flightless bird orders. The forelimbs are modified into wings.'},
 {'score': 1.08254056760404e-11,
  'start': 617,
  'end': 632,
  'answer': 'eleven segments',
  'context': 'Insects have segmented bodies supported b

### Observations

- The **retriever** uses semantic similarity so it can find relevant passages even when the question wording differs from the context text.
- The **reader** extracts an exact answer *span* from the passage rather than generating a new sentence — this is what makes it *extractive* QA.
- Using `top_k > 1` lets the reader evaluate multiple contexts; results are sorted by confidence so the best answer comes first.
- Confidence scores (`score`) close to 1.0 indicate high certainty. Low scores suggest the answer may not be in the retrieved context.
- Performance is best for questions whose answers exist in the SQuAD training corpus.

## 🧹 Step 10 — Cleanup

Delete the Pinecone index when you're done to free up resources.

In [ ]:
pc.delete_index(index_name)
print(f"Index '{index_name}' deleted successfully.")
